# 05 – Skill extraction and skill gap

Goal: for a resume and a job offer, list the skills of the offer that the resume does not mention.

1. **Vocabulary-based extraction** (`matcher/skills.py`): 218 skills: 132 technical skills kept from the 252 of the 785k `data_jobs` postings + 86 general skills, plus 56 alternative spellings; whole-word matching on cleaned text; words such as "r", "go", "spring" only count next to another technical skill.
2. **LLM extraction** (optional, needs a key in `.env`): the LLM returns a JSON list of skills. Used to measure what the vocabulary misses.
3. **Gap to the market**: for a data role, the skills asked in the most postings of `data_jobs`.

In [1]:
import sys
sys.path.insert(0, "..")          # the shared code lives in ../matcher
import numpy as np
import pandas as pd
from collections import Counter

from matcher.data import load_resumes, load_jobs, load_role_skills
from matcher.skills import extract_skills, skill_gap, role_gap, display
from matcher.llm import get_llm, llm_skills

In [2]:
resumes, jobs = load_resumes(), load_jobs()
resumes["skills"] = resumes["text"].map(extract_skills)
jobs["skills"] = jobs["text"].map(extract_skills)
print(f"Resumes: {resumes['skills'].str.len().mean():.1f} skills on average, none found in {(resumes['skills'].str.len() == 0).mean():.1%}")
print(f"Jobs:    {jobs['skills'].str.len().mean():.1f} skills on average, none found in {(jobs['skills'].str.len() == 0).mean():.1%}")

Resumes: 8.2 skills on average, none found in 3.4%
Jobs:    4.2 skills on average, none found in 8.4%


In [3]:
# Most frequent skills in each dataset
pd.DataFrame({
    "resumes": [f"{display(s)} ({n})" for s, n in Counter(s for x in resumes["skills"] for s in x).most_common(15)],
    "jobs": [f"{display(s)} ({n})" for s, n in Counter(s for x in jobs["skills"] for s in x).most_common(15)],
})

,resumes,jobs
0,communication (5879),communication (412)
1,sales (4060),sales (269)
2,leadership (3772),leadership (192)
3,testing (3562),customer service (184)
4,customer service (3423),marketing (182)
5,marketing (2541),Excel (132)
6,Excel (2420),recruitment (127)
7,project management (2405),testing (82)
8,troubleshooting (1999),project management (81)
9,SQL (1877),word (75)


In [4]:
# Do the extracted skills make sense per category? (resumes)
resumes.groupby("category")["skills"].agg(
    lambda s: ", ".join(display(k) for k, _ in Counter(x for xs in s for x in xs).most_common(6))
).loc[["Data Science", "Java Developer", "Human Resources", "Accountant", "Civil Engineer", "Sales"]]

category
Data Science       Python, communication, leadership, machine lea...
Java Developer         Java, testing, SQL, oracle, JavaScript, agile
Human Resources    recruitment, employee relations, communication...
Accountant         accounting, tax, payroll, sales, Excel, quickb...
Civil Engineer     autocad, communication, project management, le...
Sales              sales, customer service, communication, market...
Name: skills, dtype: str

In [5]:
# Ambiguous words: "r", "go", "c" are kept only next to another technical skill
tests = ["skills python r sql tableau", "i want to go to the office in spring", "languages c java python",
         "programming in r and python", "word excel powerpoint outlook", "we react quickly to customer requests"]
for t in tests:
    print(f"{t!r:45} -> {sorted(extract_skills(t))}")

'skills python r sql tableau'                 -> ['python', 'r', 'sql', 'tableau']
'i want to go to the office in spring'        -> []
'languages c java python'                     -> ['c', 'java', 'python']
'programming in r and python'                 -> ['python', 'r']
'word excel powerpoint outlook'               -> ['excel', 'outlook', 'powerpoint', 'word']
'we react quickly to customer requests'       -> []


## 1. Skill gap between a resume and an offer

In [6]:
from matcher.models import embed_documents

# A Data Science resume and its closest offer (embedding cosine, as in notebook 04)
resume = resumes[resumes["category"] == "Data Science"]["text"].iloc[3]
E_jobs = embed_documents(jobs["text"], name="jobs_chunks")
job = jobs.iloc[int(np.argmax(E_jobs @ embed_documents([resume])[0]))]
matched, missing = skill_gap(resume, job["text"])
print(f"Offer: {job['title']} ({job['company']})")
print("Skills of the offer found in the resume:", matched)
print("Missing skills:", missing)

Offer: Data Analytics Senior Mangager (Edlabs)
Skills of the offer found in the resume: ['ETL', 'data analysis', 'project management']
Missing skills: ['big data', 'data modeling', 'data visualization', 'data warehousing', 'leadership', 'oracle']


## 2. Gap to the market for a data role (`data_jobs`)

In [7]:
role_skills = load_role_skills()
print(sorted(role_skills["role"].unique()))
role_gap(resume, role_skills, "Data Analyst")

['Business Analyst', 'Cloud Engineer', 'Data Analyst', 'Data Engineer', 'Data Scientist', 'Machine Learning Engineer', 'Senior Data Analyst', 'Senior Data Engineer', 'Senior Data Scientist', 'Software Engineer']


,skill,share,in_resume
0,SQL,0.579871,False
1,Excel,0.419464,False
2,Python,0.358796,True
3,Tableau,0.291448,False
4,Power BI,0.247061,False
5,R,0.188188,True
6,SAS,0.175653,True


## 3. Vocabulary vs LLM (needs a key in `.env`)

On 20 job offers, the LLM list is taken as a reference. Each LLM skill is mapped to the vocabulary (same cleaning and aliases);
**recall** = share of the LLM skills also found by the vocabulary; skills that are not in the vocabulary at all show what it misses.

In [8]:
import time

sample = jobs.sample(15, random_state=1)
rows = []
if get_llm() is None:
    print("No API key in .env: comparison skipped")
else:
    for _, job in sample.iterrows():
        try:
            llm = llm_skills(job["description"])
        except Exception as e:                       # free tier: timeouts and rate limits happen
            print(f"skipped {job['title']}: {type(e).__name__}")
            continue
        llm_mapped = set().union(*[extract_skills(s) for s in llm]) if llm else set()
        out_of_vocab = [s for s in llm if not extract_skills(s)]
        rows.append({"title": job["title"], "vocabulary": len(job["skills"]), "LLM": len(llm),
                     "LLM skills in vocabulary": 1 - len(out_of_vocab) / max(len(llm), 1),
                     "found by vocabulary": len(llm_mapped & job["skills"]) / max(len(llm_mapped), 1),
                     "examples not in vocabulary": ", ".join(out_of_vocab[:5])})
        time.sleep(4)                                 # stay under the free-tier limit of requests per minute
comparison = pd.DataFrame(rows)
comparison

,title,vocabulary,LLM,LLM skills in vocabulary,found by vocabulary,examples not in vocabulary
0,Nurse Practitioner - Icu Davies - Pd,0,30,0.133333,0.000000,"nursing care, patient care, patient education,..."
1,Fraud Awareness Educator – Associate,6,21,0.285714,0.800000,"fraud education, training facilitation, client..."
2,Virtual administrative assistant,6,23,0.173913,1.000000,"virtual assistance, administrative support, sc..."
3,"Marketing Director – E-Learning – Columbus, OH",9,33,0.454545,0.875000,"elearning, market research, customer insights,..."
4,Talent Acquisition Specialist,1,9,0.111111,1.000000,"talent acquisition, recruitingsourcing, full l..."
5,Software Developer,3,25,0.160000,0.500000,"backend development, frontend development, rub..."
6,Data Entry Specialist,3,24,0.291667,0.500000,"loan documentation, commercial loans, monetary..."
7,Financial Professional – Financial / Insurance...,5,27,0.296296,0.625000,"finance, insurance, book of business managemen..."
8,Systems Administrator,8,47,0.148936,0.857143,"financial advisory, family office services, di..."
9,Web Designer,6,25,0.240000,0.833333,"user acquisition campaigns, paid search, socia..."


In [9]:
if len(comparison):
    print(f"{len(comparison)} offers compared")
    print(f"Skills per offer: vocabulary {comparison['vocabulary'].mean():.1f}, LLM {comparison['LLM'].mean():.1f}")
    print(f"Share of the LLM skills that exist in the vocabulary: {comparison['LLM skills in vocabulary'].mean():.0%}")
    print(f"...and that the vocabulary also found in the text: {comparison['found by vocabulary'].mean():.0%}")

15 offers compared
Skills per offer: vocabulary 5.5, LLM 26.8
Share of the LLM skills that exist in the vocabulary: 22%
...and that the vocabulary also found in the text: 77%


## Findings

- **Vocabulary extraction:** 8.2 skills per resume and 4.2 per offer on average; no skill is found in 3.4% of the resumes and 8.4% of the offers (fields outside the vocabulary: the nursing offer of the LLM comparison has 0 skill).
- The skills per category are sensible (Java Developer → Java, SQL, oracle, JavaScript, agile; Accountant → accounting, tax, payroll, QuickBooks), and the rule for ambiguous words works on the test sentences ("go to the office in spring" → no skill, "python r sql" → R).
- Three general words were removed from the vocabulary after reading the results: "training" and "reporting" are mostly section titles, "compensation" is mostly a benefit of the offer.
- **Comparison with the LLM (15 offers, Gemini):** the LLM lists 26.8 skills per offer against 5.5 for the vocabulary. Only 22% of the LLM skills exist in the vocabulary at all; for those, the vocabulary finds 77% of them in the text. The missing ones are domain skills written as phrases ("nursing care", "patient care", "fraud education").
- **Trade-off:** the vocabulary is free, instant, deterministic and explainable, but limited to known tools; the LLM covers every field but costs one API call per document, takes seconds, sends the resume to an external service and its answer can vary between runs.
- **Choice for the app:** vocabulary by default, LLM extraction as an option (checkbox) when a key is available. The gap to the market (`data_jobs`) is the most reliable output for data roles, because it is based on 785k postings.